# 08_recovery_map.ipynb

Task 4.5, phần bản đồ phục hồi, oracle và thống kê. Đo từng kỹ thuật xử lý ảnh của task 4.3 và 4.4 trên tập **test** bằng reference model ConvNeXt-Tiny đóng băng, cùng 888 ảnh và nhãn, theo kỹ thuật × họ suy giảm × mức.

Dùng đúng các cấu hình đã khóa trên validation (`reports/processing/*_selected.json`), không chỉnh gì theo kết quả test. Quy tắc thống kê và "có bằng chứng" khóa trong `configs/recovery_map.yaml`.

Hai giai đoạn trong một phiên:
- **A (GPU):** chạy model trên ảnh suy giảm không xử lý và đã xử lý, lưu dự đoán và xác suất từng ảnh. Mục này lâu nhất.
- **B (CPU):** bảng recovery với khoảng tin cậy bootstrap, McNemar ghép cặp có hiệu chỉnh Holm, chi phí trên ảnh sạch, hai kiểu oracle, và so với model huấn luyện lại (kết quả của notebook 07).

Test chỉ mở một lần, cần `FINAL_EVAL=1`.

## 1. Môi trường

Internet để On. Accelerator chọn GPU (T4). Dataset input cần hai cái: `cthng123/durian-leaf-disease-dataset` và dataset chứa checkpoint `convnext_tiny_seed42.pt` (cùng dataset đã dùng cho notebook 03 đến 06).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Đổi nhánh tại đây, rồi chạy lại cell.
GIT_BRANCH = "recovery-map"
REPO_URL = "https://github.com/meo225/durian-leaf-disease-robustness.git"
REPO_DIR = Path("/kaggle/working") / "durian-leaf-disease-robustness"

if not (Path("/kaggle/working").is_dir() and Path("/kaggle/input").is_dir()):
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle. Xem README, mục Chạy trên Kaggle.")

def git(args, cwd=None):
    subprocess.run(
        ["git", *args],
        cwd=cwd,
        check=True,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )


if not (REPO_DIR / ".git").is_dir():
    git(["clone", "--branch", GIT_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    git(["fetch", "origin", GIT_BRANCH], cwd=REPO_DIR)
    git(["checkout", GIT_BRANCH], cwd=REPO_DIR)
    git(["reset", "--hard", f"origin/{GIT_BRANCH}"], cwd=REPO_DIR)

matches = []
seen = set()
for path in Path("/kaggle/input").rglob("DLD_FinalDataset_224_spit"):
    resolved = str(path.resolve())
    if path.is_dir() and resolved not in seen:
        seen.add(resolved)
        matches.append(path)

if not matches:
    found = []
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for path in sorted(input_root.rglob("*")):
            if path.is_dir() and len(path.relative_to(input_root).parts) <= 3:
                found.append(str(path.relative_to(input_root)))
    listing = "\n".join(found[:40]) if found else "(trống)"
    raise FileNotFoundError(
        "Chưa thấy DLD_FinalDataset_224_spit trong /kaggle/input.\n"
        "Add Input: cthng123/durian-leaf-disease-dataset, rồi Save & Run All.\n"
        f"Thư mục hiện có:\n{listing}"
    )

image_root = matches[0].parent
link = REPO_DIR / "datasets" / "durian-ldd"
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink() or link.is_file():
    link.unlink()
elif link.exists():
    link.rmdir()
link.symlink_to(image_root, target_is_directory=True)

if not (link / "DLD_FinalDataset_224_spit").is_dir():
    raise FileNotFoundError(f"Không thấy ảnh tại {link}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True
).strip()
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, text=True
).strip()
print(f"Nhánh: {branch}")
print(f"Commit: {commit}")
print(f"Ảnh: {link / 'DLD_FinalDataset_224_spit'}")

## 2. Cấu hình, checkpoint và dữ liệu

Kiểm tra GPU, đọc cấu hình, nạp reference model. Thiếu GPU thì dừng ngay.

In [ ]:
import json
import os
import time

import cv2
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import Image, display

from src.corruption import load_conditions, load_seed
from src.evaluation.recovery_map import load_selected, run_map, save_run, task_scopes
from src.evaluation.recovery_report import build_report
from src.evaluation.reference import collect_checkpoint, load_trained_model
from src.training.loop import resolve_device

CONFIG = "configs/recovery_map.yaml"
cfg = yaml.safe_load((REPO_DIR / CONFIG).read_text(encoding="utf-8"))
pipeline_cfg = yaml.safe_load((REPO_DIR / "configs/pipeline.yaml").read_text(encoding="utf-8"))
classes = list(pipeline_cfg["classes"])
RES = REPO_DIR / "outputs/results/recovery_map"
REP = REPO_DIR / cfg["output_dir"]
for folder in (RES, REP):
    folder.mkdir(parents=True, exist_ok=True)

device = resolve_device()
if device.type != "cuda":
    raise RuntimeError("Notebook này cần GPU. Trong Session options chọn Accelerator GPU, rồi chạy lại từ cell đầu.")
# Bộ lọc OpenCV giải phóng GIL nên chạy song song bằng luồng. Mỗi luồng dùng một nhân để khỏi tranh nhau.
WORKERS = max(1, min(8, os.cpu_count() or 1))
cv2.setNumThreads(1)

reference = json.loads((REPO_DIR / "reports/baselines/reference_selection.json").read_text(encoding="utf-8"))["selection"]["reference"]
assert reference == "convnext_tiny", f"Reference model là {reference}, notebook này viết cho convnext_tiny."
train_cfg = yaml.safe_load((REPO_DIR / cfg["train_config"]).read_text(encoding="utf-8"))
checkpoint = collect_checkpoint("/kaggle/input", cfg["checkpoint_name"], "/kaggle/working/checkpoints")
model = load_trained_model(train_cfg, checkpoint, len(classes), device)

selected = load_selected(cfg["selected"])
scopes = None if cfg["scope"] == "all" else task_scopes(cfg["task_configs"])
os.environ["FINAL_EVAL"] = "1"  # mở tập test, đúng một lần cho bản đồ này
seed = load_seed(cfg["corruption_config"])
conditions = load_conditions(cfg["corruption_config"], cfg["split"])  # từ chối nếu config suy giảm chưa khóa
print(f"Thiết bị: {device}, reference: {reference}, số luồng lọc ảnh: {WORKERS}")
print(f"Kỹ thuật: {', '.join(selected)} ({len(selected)})")
print(f"Phạm vi: {cfg['scope']}, {len(conditions)} điều kiện trên tập {cfg['split']}")

## 3. Giai đoạn A: chạy bản đồ trên test

Với mỗi điều kiện, ảnh suy giảm tạo một lần rồi mọi kỹ thuật chạy lên chính ảnh đó. Ảnh sạch cũng chạy qua mọi cấu hình để đo chi phí. Dự đoán lưu sau mỗi điều kiện, nên nếu phiên bị ngắt vẫn còn bản dở dang trong `outputs/results/recovery_map/`.

In [ ]:
start = time.time()
partial = RES / "recovery_predictions_test_partial.npz"
log = []


def progress(message):
    log.append(message)
    print(f"[{(time.time() - start) / 60:5.1f} phút] {message}", flush=True)


result = run_map(
    model, pipeline_cfg, cfg["split"], conditions, selected, device, seed,
    scopes=scopes, workers=WORKERS, progress=progress,
    on_condition_done=lambda partial_result: save_run(partial_result, partial),
)
run_path = RES / "recovery_predictions_test.npz"
save_run(result, run_path)
partial.unlink(missing_ok=True)
print(f"Giai đoạn A xong sau {(time.time() - start) / 60:.1f} phút: {len(result['predictions'])} lượt chạy, {len(result['labels'])} ảnh")
print(f"Đã lưu {run_path} ({run_path.stat().st_size / 1024**2:.1f} MB)")

## 4. Giai đoạn B: bảng, khoảng tin cậy và so sánh

Kiểm tra trước: dự đoán "không xử lý" ở đây phải trùng với reference ở task 4.2 trên cùng ảnh (`mismatch_vs_task_4_2`, kỳ vọng 0 hoặc rất nhỏ do làm tròn xác suất). Sau đó tính recovery, McNemar có hiệu chỉnh Holm, chi phí ảnh sạch, oracle và so với model huấn luyện lại.

In [ ]:
built = build_report(CONFIG, run_path)
summary = built["summary"]
print(f"Cùng mẫu với task 4.2: {summary['same_samples_as_task_4_2']}, số dự đoán lệch so với 4.2: {summary['mismatch_vs_task_4_2']['total']} (tối đa {summary['mismatch_vs_task_4_2']['max_per_condition']} mỗi điều kiện)")
print(f"{summary['cells']} ô trong bản đồ phục hồi")
display(built["strategies"].round(4))

### Cặp (kỹ thuật, họ) có bằng chứng

Theo quy tắc khóa trước: khoảng tin cậy của recovery trung bình ba mức nằm trên 0, và ít nhất hai mức có recovery dương có ý nghĩa sau hiệu chỉnh Holm. Cột `costly` đánh dấu cấu hình mà cận trên khoảng tin cậy thay đổi F1 ảnh sạch thấp hơn mức ngưỡng. Đây là danh sách đầu vào của Phase 5.

In [ ]:
family = built["family"]
columns = ["technique", "family", "config_id", "recovery_mean", "recovery_low", "recovery_high", "levels_gain_significant", "clean_delta", "evidence", "costly"]
display(family[family["suitable"]][columns].round(3))
print("Có bằng chứng:", ", ".join(f"{row.technique}/{row.family}" for row in family[family["evidence"]].itertuples()) or "không có")

### Chi phí trên ảnh sạch

Thay đổi macro F1 trên 888 ảnh sạch khi áp từng cấu hình, kèm khoảng tin cậy và p đã hiệu chỉnh Holm.

In [ ]:
display(built["clean_cost"][["technique", "config_id", "used_for_families", "f1_clean", "f1_processed", "delta", "delta_low", "delta_high", "p_holm"]].round(4))

### Bản đồ phục hồi và so sánh chiến lược

Heatmap recovery và normalized recovery (ô khung đen là cặp phù hợp), rồi macro F1 theo họ của không xử lý, oracle chọn theo họ trên validation và model huấn luyện lại.

In [ ]:
for name in ["recovery_heatmap", "normalized_recovery_heatmap", "strategies_by_family"]:
    print(name)
    display(Image(filename=str(REP / f"{name}.png")))

## 5. Lưu kết quả

Nén toàn bộ thư mục kết quả. Bản chốt nằm ở `reports/recovery_map/` (bảng CSV, JSON tóm tắt, hình). File dự đoán từng ảnh (`recovery_predictions_test.npz`) nằm trong output Kaggle.

In [ ]:
import zipfile

archive = Path("/kaggle/working/recovery_map_outputs.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for folder in (RES, REP):
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                bundle.write(path, path.relative_to(REPO_DIR).as_posix())
print(f"Đã nén {archive} ({archive.stat().st_size / 1024**2:.1f} MB)")
print("Bản chốt: reports/recovery_map/. Dự đoán từng ảnh: outputs/results/recovery_map/recovery_predictions_test.npz")